# Deploy Approved Model to SageMaker Real-Time Endpoint

In [ ]:
%pip install "sagemaker<3.0.0" "prettytable>=3.9" --upgrade -qqqq

In [ ]:
%env SAGEMAKER_SUPPRESS_V2_WARNING=1

In [ ]:
import json
import time
import boto3
import pandas as pd
import numpy as np
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model import ModelPackage
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer, JSONSerializer
from sagemaker.deserializers import JSONDeserializer, CSVDeserializer
from sagemaker.model_monitor import DataCaptureConfig

In [ ]:
session = sagemaker.Session()
bucket = session.default_bucket()

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(
    service_name="sagemaker",
    region_name=region
)

model_package_group_name = "CarePath-Diabetes-Readmission"
endpoint_name = f"carepath-readmission-endpoint-{int(time.time())}"
data_capture_prefix = "carepath-analytics/datacapture"
data_capture_s3_uri = f"s3://{bucket}/{data_capture_prefix}"
role = get_execution_role()

print(f"AWS Region: {region}")
print(f"Default Bucket: {bucket}")
print(f"Target Model Package Group: {model_package_group_name}")
print(f"Target Endpoint Name: {endpoint_name}")

## 1. Retrieve the Latest Approved Model Package

In [ ]:
# Get latest approved package
response = sagemaker_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    ModelApprovalStatus="Approved",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)

packages = response.get("ModelPackageSummaryList", [])

# If no approved packages, get the latest package regardless of approval
if not packages:
    response = sagemaker_client.list_model_packages(
        ModelPackageGroupName=model_package_group_name,
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=1,
    )
    packages = response.get("ModelPackageSummaryList", [])
    print(f"No packages with 'Approved' status found. Defaulting to latest package, regardless of approval")

if not packages:
    raise ValueError(f"No model in model package group: {model_package_group_name}")

latest_package_arn = packages[0]["ModelPackageArn"]
print(f"Package ARN: {latest_package_arn}")
print(f"Created: {packages[0]['CreationTime']}")

## 2. Configure Real-Time Endpoint with Data Capture
Use **DataCaptureConfig** to record request and response payloads to S3 for **SageMaker Model Monitor** and **CloudWatch**

In [ ]:
# Load model
model = ModelPackage(
    role=role,
    model_package_arn=latest_package_arn,
    sagemaker_session=session,
)

# Define request/response capture config
data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=data_capture_s3_uri,
    capture_options=["REQUEST", "RESPONSE"],
    csv_content_types=["text/csv"],
    json_content_types=["application/json"],
    sagemaker_session=session,
)

print(f"Deploying {endpoint_name}...")

# Deploy real-time endpoint
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config,
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
)

print(f"Endpoint successfully deployed: {predictor.endpoint_name}")

## 3. Real-Time Endpoint Testing

In [ ]:
request_low_risk = [
    []
]

request_high_risk = [
    []
]

# Configure Predictor serializers
predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

# Predict low risk patient
pred_low = predictor.predict(request_low_risk)
print("Low-Risk Readmission Test ---")
print(f"Response: {pred_low}")

# Predict high risk patient
pred_high = predictor.predict(request_high_risk)
print("\nHigh-Risk Readmission Test ---")
print(f"Response: {pred_high}")